# Phase 3 — baseline GraphCast_small forecasts of the case studies (Colab GPU)

**Runtime:** Runtime → Change runtime type → **T4 GPU**.
Order: 1 code → 2 install (+ restart) → 3 check GPU → 4 storage & model → 5 input check →
6 forecasts (timing test, then all) → 7 no-harm → 8 copy results to the Mac.
Steps 6 and 7 are resumable: after a disconnect, run 1, 3, 4 again and rerun the step.

## 1. Get the code

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/RomanGorelsky/graphcast-extremes"
if os.path.exists("/content/graphcast-extremes"):
    subprocess.run(["git", "-C", "/content/graphcast-extremes", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/graphcast-extremes"], check=True)
%cd /content/graphcast-extremes
!git log -1 --format="%h %s"

## 2. Install (≈4 min), then restart

Installs the pinned requirements, swaps in the GPU build of JAX 0.7.2 and removes Colab's
newer CUDA-13 plugins. Afterwards: **Runtime → Restart session**, then continue with §3
(no reinstall needed after a restart).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!pip uninstall -y -q jax-cuda13-plugin jax-cuda13-pjrt
!pip install -q "jax[cuda12]==0.7.2"
print("Now: Runtime -> Restart session, then run section 3")

## 3. After the restart: check the GPU

In [ ]:
import os, subprocess, jax
%cd /content/graphcast-extremes
print(jax.__version__, jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU - Runtime -> Change runtime type -> T4 GPU"

## 4. Storage and model files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["EXTREMES_DATA"] = "/content/data"
os.environ["EXTREMES_RESULTS"] = "/content/drive/MyDrive/graphcast-extremes/results"
!python scripts/00_download_assets.py
!python -m pytest -q tests

## 5. Are our ERA5 inputs identical to the model's own data?

Must end with `ALL INPUTS MATCH`. Otherwise stop and send the output to Claude.

In [ ]:
!python scripts/30_check_inputs.py

## 6. Case-study forecasts (56 × 10 days)

First one event (4 forecasts) as a timing test; then all. Rerun the second cell after any
disconnect — finished forecasts are skipped.

In [ ]:
!python scripts/31_run_case_forecasts.py --only heat-004

In [ ]:
!python scripts/31_run_case_forecasts.py

## 7. 'Do no harm' reference (12 ordinary dates in 2020)

In [ ]:
!python scripts/32_run_noharm.py --label R0

## 8. Bring the results to the Mac

From `MyDrive/graphcast-extremes/results/phase3/` download into
`~/Documents/GitHub/graphcast-extremes/results/phase3/`:
the whole `forecasts/` folder (56 small files), `noharm.csv`, `input_check.json`.
Then on the Mac: `python scripts/33_verify.py`.